# Prophet Modeling for Commodity Prices

In this notebook, we focus on **time series forecasting of commodity prices** using **Facebook Prophet**.  
Prophet is designed to handle seasonality, trends, and outliers effectively, making it suitable for commodity price data.

The workflow includes:
1. Loading and preparing the data  
2. Filtering and formatting for Prophet  
3. Splitting into train and test sets  
4. Prophet model fitting and evaluation  
5. Forecasting and diagnostics

In [ ]:
# Data manipulation and numerical operations
import pandas as pd       
import numpy as np        

# Model persistence and optimization
import pickle             
import optuna    
         
# Time series forecasting
from prophet import Prophet                         
from prophet.diagnostics import cross_validation, performance_metrics  

# Model evaluation metrics
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

# Handling outputs during tuning or diagnostics
import io                
from contextlib import redirect_stdout, redirect_stderr 

# Custom utilities for data handling and preprocessing
from utils.load_data import load_data                
from utils.data_splitter import DataSplitter        
from utils.model_filters import filter_for_prophet  

# Pandas display settings for full visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Reduce verbosity of Optuna during hyperparameter tuning
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Load the commodity price dataset
df = load_data('data/commodity_prices.csv')

In [2]:
# Define the train, validation, and test periods for modeling
date_dict = {
    'train_start': "2023-06-01", 'train_end': "2025-06-30",
    'valid_start': "2025-07-01", 'valid_end': "2025-07-31",
    'test_start': "2025-08-01", 'test_end': "2025-08-18"
}

# Set minimum data thresholds for each split to ensure sufficient observations
prophet_thresholds = {
    'train': 250,
    'valid': 10,
    'test': 5
}

# Initialize the custom splitter to generate train, validation, and test sets
splitter = DataSplitter(df, date_dict, prophet_thresholds)
train_df, valid_df, test_df = splitter.run()

# Apply Prophet-specific filtering and cutoff to remove sparse or incomplete series
train_df, valid_df, test_df = filter_for_prophet(train_df, valid_df, test_df, cutoff="2025-06-15")

# Prophet Forecasting per Product-Market Pair

In this section, we train **individual Prophet models** for each unique combination of `Product_Type` and `Market`.  
To ensure accurate and robust forecasts, we perform:

- **Hyperparameter tuning using Optuna**: Optimizes changepoint flexibility, seasonality strength, and seasonality mode.  
- **Cross-validation within Prophet**: Evaluates model performance with a rolling forecast to compute RMSE.  
- **Silent fitting and logging suppression**: Prevents verbose output from Prophet and CmdStanPy during optimization.  
- **Model persistence**: Saves trained models for future use and comparison.  

The final output is a **DataFrame summarizing the best hyperparameters and RMSE** for each product-market pair, along with saved models ready for forecasting.

In [3]:
# Rename columns for Prophet compatibility
train_df = train_df.rename(columns={
    'Arrival_Date': 'ds',    # Prophet requires 'ds' for date
    'log_Modal_Price': 'y'   # Prophet requires 'y' for target variable
})

# Get all unique (Product_Type, Market) pairs for separate models
pairs = train_df[['Product_Type', 'Market']].drop_duplicates()
results = []

# Iterate over each product-market pair to fit individual Prophet models
for _, row in pairs.iterrows():
    product = row['Product_Type']
    market = row['Market']

    # Subset data for the current pair
    subset = train_df[(train_df['Product_Type'] == product) & (train_df['Market'] == market)]
    prophet_df = subset[['ds', 'y']]

    # Define Optuna objective function for hyperparameter tuning
    def objective(trial):
        model = Prophet(
            changepoint_prior_scale=trial.suggest_float('changepoint_prior_scale', 0.001, 0.5, log=True),
            seasonality_prior_scale=trial.suggest_float('seasonality_prior_scale', 1, 20, log=True),
            seasonality_mode=trial.suggest_categorical("seasonality_mode", ["additive", "multiplicative"]),
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False
        )

        # Suppress verbose Prophet/Stan logs during fitting
        f = io.StringIO()
        with redirect_stdout(f), redirect_stderr(f):
            model.fit(prophet_df)

            df_cv = cross_validation(
                model,
                initial='365 days',
                period='90 days',
                horizon='30 days',
                disable_tqdm=True  # disables progress bar
            )

        # Return mean RMSE across cross-validation folds
        df_perf = performance_metrics(df_cv)
        return df_perf['rmse'].mean()

    # Run Optuna hyperparameter optimization
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=30, show_progress_bar=False, n_jobs=1)

    best_params = study.best_params
    best_rmse = study.best_value

    # Fit final Prophet model using optimized hyperparameters
    f = io.StringIO()
    with redirect_stdout(f), redirect_stderr(f):
        best_model = Prophet(
            **best_params,
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False
        )
        best_model.fit(prophet_df)

    # Save trained model for future use
    filename = f"models/prophet/prophet_{product}_{market}.pkl"
    with open(filename, "wb") as f_model:
        pickle.dump(best_model, f_model)

    # Store results for analysis
    results.append({
        "Product_Type": product,
        "Market": market,
        "best_params": best_params,
        "best_rmse": best_rmse
    })

# Compile all results into a DataFrame for review
results_df = pd.DataFrame(results)

# Validation of Prophet Models

In this section, we evaluate the **pre-trained Prophet models** on the **validation set**:

- Each `Product_Type` and `Market` pair is forecasted using its corresponding trained model.  
- Predictions are aligned with actual log-prices from the validation period.  
- Performance metrics computed include:
  - **RMSE** – measures average prediction error magnitude.  
  - **MAE** – captures mean absolute deviation.  
  - **MAPE** – expresses average prediction error as a percentage.  

The results are compiled into `val_results_df`, providing a clear overview of **forecast accuracy** across products and markets.

In [4]:
# Rename columns for Prophet compatibility
valid_df = valid_df.rename(columns={
    'Arrival_Date': 'ds',
    'log_Modal_Price': 'y'
})

val_results = []

# Get all unique product-market pairs in the validation set
pairs = valid_df[['Product_Type', 'Market']].drop_duplicates()

# Evaluate each Prophet model on its corresponding validation series
for _, row in pairs.iterrows():
    product = row['Product_Type']
    market = row['Market']

    # Load the pre-trained Prophet model
    filename = f"models/prophet/prophet_{product}_{market}.pkl"
    try:
        with open(filename, "rb") as f:
            model = pickle.load(f)
    except FileNotFoundError:
        print(f"Model not found for {product}, {market}, skipping...")
        continue

    # Subset validation data for the current pair
    subset = valid_df[(valid_df['Product_Type'] == product) & (valid_df['Market'] == market)]
    
    # Generate forecast for the validation period
    future = model.make_future_dataframe(periods=60)
    forecast = model.predict(future)

    # Extract predictions corresponding to validation dates
    forecast_val = forecast[forecast['ds'].isin(subset['ds'])]
    merged = forecast_val[["ds", "yhat"]].merge(subset[["ds", "y"]], on="ds")

    # Compute validation metrics
    rmse = root_mean_squared_error(merged["y"], merged["yhat"])
    mae = mean_absolute_error(merged["y"], merged["yhat"])
    mape = np.mean(np.abs((merged["y"] - merged["yhat"]) / merged["y"])) * 100

    # Store results for review
    val_results.append({
        "Product_Type": product,
        "Market": market,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape
    })

# Compile validation metrics into a DataFrame
val_results_df = pd.DataFrame(val_results)

# Prophet Hyperparameter Tuning Results

The table below summarizes the **best-performing Prophet models** for each product-market pair:

- `Product_Type` and `Market` indicate the series modeled.  
- `best_params` contains the optimized hyperparameters found via **Optuna**.  
- `best_rmse` reports the **mean RMSE** from cross-validation, reflecting model accuracy.

This allows quick comparison of performance across different commodities and markets, and identifies series that may require further attention.

In [5]:
results_df.head()

,Product_Type,Market,best_params,best_rmse
0,Alsandikai|Alsandikai|FAQ,North Paravur,{'changepoint_prior_scale': 0.0162409674144847...,0.090810
1,Amaranthus|Amaranthus|FAQ,Aluva,{'changepoint_prior_scale': 0.0078154401650040...,0.123889
2,Amaranthus|Amaranthus|FAQ,Angamaly,{'changepoint_prior_scale': 0.0046421196839204...,0.098430
3,Amaranthus|Amaranthus|FAQ,Broadway market,{'changepoint_prior_scale': 0.2988485760895735...,0.067958
4,Amaranthus|Amaranthus|FAQ,Ernakulam,{'changepoint_prior_scale': 0.0078419796556634...,0.112590


# Validation Results for Prophet Models

The `val_results_df` summarizes **model performance on the validation set** for each product-market pair:

- `Product_Type` and `Market` identify the series evaluated.  
- `y_true` and `y_pred` show actual vs. predicted log-prices.  
- Metrics such as RMSE or MAE quantify **forecast accuracy** on unseen data.  

This table helps assess **generalization** of the Prophet models before testing on the final holdout period.

In [6]:
val_results_df.head()

,Product_Type,Market,RMSE,MAE,MAPE
0,Alsandikai|Alsandikai|FAQ,North Paravur,0.362889,0.335285,3.966713
1,Amaranthus|Amaranthus|FAQ,Aluva,0.258197,0.213394,2.570218
2,Amaranthus|Amaranthus|FAQ,Angamaly,0.110461,0.109474,1.367274
3,Amaranthus|Amaranthus|FAQ,Broadway market,0.052545,0.043910,0.544033
4,Amaranthus|Amaranthus|FAQ,Ernakulam,0.156060,0.113095,1.391658
